# Inequality Enumeration: D1 -> D2 Pipeline (Paper-First)

This notebook enumerates the extremal inequalities produced by the oracle pipeline for the
paper four-tree family $(T_0, T_1, T_2, T_3)$ at $n=3$, $m=4$, and matches the paper's
named inequalities -- ECP (5.4), (5.5), and Remark 5.5 -- to specific extreme rays by
coefficient-vector signature.

In [1]:
using Pkg
Pkg.activate(joinpath(@__DIR__, "..", "PercolationOracle"))
Pkg.develop(path=joinpath(@__DIR__, "..", "ProjectedConeOracle"))
Pkg.instantiate()

using PercolationOracle
using ProjectedConeOracle
using Printf

function enumerate_quiet(n_obs, m; kwargs...)
    result = redirect_stdout(devnull) do
        redirect_stderr(devnull) do
            enumerate_all_inequalities(n_obs, m; verbose=false, kwargs...)
        end
    end
    return result
end

println("Julia version: ", VERSION)
println("PercolationOracle: ", pathof(PercolationOracle))
println("ProjectedConeOracle: ", pathof(ProjectedConeOracle))
println("LP solver: HiGHS (default)")

  Activating 

project at `~/Documents/Thesis/bunkbed-oracle-packages/PercolationOracle`


   Resolving 

package versions...


  No Changes to `~/Documents/Thesis/bunkbed-oracle-packages/PercolationOracle/Project.toml`
  No Changes to `~/Documents/Thesis/bunkbed-oracle-packages/PercolationOracle/Manifest.toml`


Julia version: 1

.11.6
PercolationOracle: /Users/azimin/Documents/Thesis/bunkbed-oracle-packages/PercolationOracle/src/PercolationOracle.jl
ProjectedConeOracle: /Users/azimin/Documents/Thesis/bunkbed-oracle-packages/ProjectedConeOracle/src/ProjectedConeOracle.jl
LP solver: HiGHS (default)


## 1. Canonical Notation (ECP / Chapter 4)

The partition set for $n=3$ is

$$\mathcal{J}_3 = \{abc,\; a|b|c,\; a|bc,\; ab|c,\; ac|b\}.$$

The mapping between ECP notation and the package's internal labels:

| ECP | Internal |
|-----|----------|
| abc | 123 |
| a\|b\|c | 1\|2\|3 |
| a\|bc | 1\|23 |
| ab\|c | 12\|3 |
| ac\|b | 13\|2 |

Event shorthand identities:

- $\mu(ab \cup ac) = \mu(abc) + \mu(ab|c) + \mu(ac|b)$
- $\mu(a|b \cap a|c) = \mu(a|b|c) + \mu(a|bc)$

After this point, the notebook uses ECP notation in narrative and the internal labels only
for programmatic operations.

In [2]:
n_obs = 3
println("Public partition order for n=$n_obs (paper convention):")
for (i, label) in enumerate(partition_order_labels(n_obs))
    println("  $i. $label")
end

Public partition order for n=3 (paper convention):


  1. 123
  2. 1|2|3
  3. 1|23
  4. 12|3
  5. 13|2


## 2. Feasible Tuple Set

For the paper four-tree family $(T_0, T_1, T_2, T_3)$, the oracle enumerates all feasible
partition pairs. The archive is normalized to paper tree order on load.

In [3]:
tuples = load_feasible_tuples()

println("Feasible set sizes |F| by prefix length m:")
for m in 1:4
    projected = Set(Tuple(t[i] for i in 1:m) for t in tuples)
    println("  m = $m: |F| = $(length(projected))")
end

println("\nArchive path: ", PercolationOracle.PAPER_FAMILY_F_PATH)
println("Note: Raw archive uses legacy tree order; package normalizes to (T_0,T_1,T_2,T_3) on load.")

Feasible set sizes |F| by prefix length m:


  m = 1: |F| = 25


  m = 2: |F| = 139
  m = 3: |F| = 570


  m = 4: |F| = 1265



Archive path: /Users/azimin/Documents/Thesis/bunkbed-oracle-packages/PercolationOracle/data/valid_partition_tuples_nobs3_notebook.jls
Note: Raw archive uses legacy tree order; package normalizes to (T_0,T_1,T_2,T_3) on load.


## 3. Run the D1 -> D2 Pipeline

In [4]:
result_m2 = enumerate_quiet(3, 2)
println("m=2 sanity check: $(length(result_m2.rays)) rays, $(length(result_m2.facet_normals)) facets ($(round(result_m2.timing, digits=2))s)")

m=2 sanity check: 15 rays, 15 facets (0.95s)


In [5]:
result_m4 = enumerate_quiet(3, 4)
pr = result_m4.projection_result

println("m=4 results (paper family):")
println("  Converged:       $(pr.converged)")
println("  |F| =            1265")
println("  Extreme rays:    $(length(result_m4.rays))")
println("  Facet normals:   $(length(result_m4.facet_normals))")
println("  Timing:          $(round(result_m4.timing, digits=2))s")
println("  Iterations:      $(pr.iterations)")
println("  LP calls:        $(pr.lp_calls)")

m=4 results (paper family):
  Converged:       true
  |F| =            1265
  Extreme rays:    17
  Facet normals:   19
  Timing:          3.7s
  Iterations:      2
  LP calls:        21


## 4. All 17 Extremal Inequalities (Human-Readable View)

The strings below are a convenience rendering. Matching named inequalities by parsing these
strings would be brittle (formatting can change when coordinate order, canonicalization, or
sign conventions change). Section 6 below uses coefficient-vector signatures instead.

In [6]:
println("All $(length(result_m4.formatted_inequalities)) extremal inequalities for m=4:\n")
for (i, ineq) in enumerate(result_m4.formatted_inequalities)
    println("  ($i)  $ineq")
end

All 17 extremal inequalities for m=4:

  (1)  mu(123)*mu(1|2|3) - mu(1|23)*mu(12|3) - mu(1|23)*mu(13|2) - mu(12|3)*mu(13|2) >= 0
  (2)  mu(1|2|3)*mu(13|2) >= 0
  (3)  mu(1|2|3)*mu(12|3) >= 0
  (4)  mu(1|2|3)*mu(1|23) >= 0
  (5)  mu(123)*mu(1|23) >= 0
  (6)  mu(123)*mu(13|2) >= 0
  (7)  mu(123)*mu(12|3) >= 0
  (8)  mu(1|23)*mu(13|2) >= 0
  (9)  mu(1|2|3)^2 >= 0
  (10)  mu(1|23)^2 >= 0
  (11)  mu(12|3)*mu(13|2) >= 0
  (12)  mu(13|2)^2 >= 0
  (13)  mu(12|3)^2 >= 0
  (14)  mu(1|23)*mu(12|3) >= 0
  (15)  -mu(123)*mu(1|2|3) + mu(123)*mu(1|23) + mu(123)*mu(12|3) + mu(123)*mu(13|2) + mu(1|2|3)*mu(1|23) + mu(1|23)*mu(12|3) + mu(1|23)*mu(13|2) + 2*mu(12|3)*mu(13|2) >= 0
  (16)  mu(1|23)^2 - mu(123)*mu(1|2|3) + mu(123)*mu(12|3) + mu(123)*mu(13|2) + mu(1|2|3)*mu(1|23) + mu(1|23)*mu(12|3) + mu(1|23)*mu(13|2) + 2*mu(12|3)*mu(13|2) >= 0
  (17)  mu(123)^2 >= 0


## 5. The Paper's Named Inequalities

**ECP (4.3)** -- Inequality (7) / Proposition 10.1:

$$\mu(a|b \cap a|c)\,\mu(ab \cup ac) \le \mu(ab|c) + \mu(ac|b) + \mu(a|bc).$$

**ECP (5.4)** -- Computer-assisted inequality / Equation (11):

$$\mu(a|b \cap a|c)\,\mu(ab \cup ac) \le \mu(ab|c) + \mu(ac|b) + \mu(a|bc) - \mu(ab|c)^2 - \mu(ac|b)^2.$$

**ECP (5.5)** -- Aas inequality / Equation (12):

$$\mu(abc)\,\mu(a|b|c) \ge \mu(ab|c)\,\mu(ac|b) + \mu(ab|c)\,\mu(a|bc) + \mu(ac|b)\,\mu(a|bc).$$

**ECP Remark 5.5** -- Third extremal ray / Ray 15:

$$\mu(ab \cup ac)\,\mu(a|b \cap a|c) + \mu(a|bc)^2 + \mu(ac|b)^2 + \mu(ab|c)^2 \le \mu(a|bc) + \mu(ac|b) + \mu(ab|c) + \mu(abc)\,\mu(a|bc).$$

## 6. Homogenization to Symmetric Coordinates

The projected-cone rays are homogeneous quadratic forms:
$\sum_{p \le q} \Phi(p,q)\,\mu(p)\mu(q) \ge 0.$

But (4.3), (5.4), and Remark 5.5 have linear terms. To convert, replace every linear
term $L(\mu)$ by $(\sum_p \mu(p)) \cdot L(\mu)$, using $\sum_p \mu(p) = 1$, then expand.

Let $x = \mu(abc)$, $y = \mu(a|b|c)$, $z = \mu(a|bc)$, $u = \mu(ab|c)$,
$v = \mu(ac|b)$, $S = x + y + z + u + v = 1$.

**ECP (5.5)** is already homogeneous:

$$xy - uv - uz - vz \ge 0$$

**ECP (5.4)** homogenized:

$$-xy + yz + z^2 + xu + xv + zu + zv + 2uv \ge 0$$

**ECP Remark 5.5** homogenized:

$$-xy + xz + yz + xu + xv + zu + zv + 2uv \ge 0$$

**ECP (4.3)** homogenized:

$$-xy + yz + z^2 + xu + xv + zu + zv + 2uv + u^2 + v^2 \ge 0$$

Key observations:

- (5.4) and Remark 5.5 differ only in the $z^2$ vs $xz$ term
- (4.3) = (5.4) + $u^2 + v^2$ -- so **(4.3) is not extremal** (it is a sum of (5.4) and
  two trivial positivity rays)

## 7. Matching Named Inequalities to Rays by Signature

Each extreme ray is a coefficient vector in the 15-dimensional symmetric coordinate space.
We construct the expected coefficient vector for each named inequality from the homogenized
forms above, canonicalize to primitive integers, and search for an exact match among the
17 enumerated rays.

In [7]:
# Build label-to-index map from result labels
label_idx = Dict(l => i for (i, l) in enumerate(result_m4.labels))

# Helper: construct a coefficient vector from named monomials
function make_sig(pairs::Pair{String,Int}...)
    v = zeros(Int, length(result_m4.labels))
    for (l, c) in pairs
        haskey(label_idx, l) || error("Unknown label: $l")
        v[label_idx[l]] = c
    end
    return canonicalize_integer_ray(v; normalize_sign=false)
end

# ECP (5.5) Aas: xy - uz - vz - uv >= 0
sig_55 = make_sig(
    "mu(123)*mu(1|2|3)" => 1,
    "mu(1|23)*mu(12|3)" => -1,
    "mu(1|23)*mu(13|2)" => -1,
    "mu(12|3)*mu(13|2)" => -1,
)

# ECP (5.4): -xy + yz + z^2 + xu + xv + zu + zv + 2uv >= 0
sig_54 = make_sig(
    "mu(123)*mu(1|2|3)" => -1,
    "mu(1|2|3)*mu(1|23)" => 1,
    "mu(1|23)^2" => 1,
    "mu(123)*mu(12|3)" => 1,
    "mu(123)*mu(13|2)" => 1,
    "mu(1|23)*mu(12|3)" => 1,
    "mu(1|23)*mu(13|2)" => 1,
    "mu(12|3)*mu(13|2)" => 2,
)

# ECP Remark 5.5: -xy + xz + yz + xu + xv + zu + zv + 2uv >= 0
sig_r55 = make_sig(
    "mu(123)*mu(1|2|3)" => -1,
    "mu(123)*mu(1|23)" => 1,
    "mu(1|2|3)*mu(1|23)" => 1,
    "mu(123)*mu(12|3)" => 1,
    "mu(123)*mu(13|2)" => 1,
    "mu(1|23)*mu(12|3)" => 1,
    "mu(1|23)*mu(13|2)" => 1,
    "mu(12|3)*mu(13|2)" => 2,
)

# ECP (4.3): -xy + yz + z^2 + xu + xv + zu + zv + 2uv + u^2 + v^2 >= 0
sig_43 = make_sig(
    "mu(123)*mu(1|2|3)" => -1,
    "mu(1|2|3)*mu(1|23)" => 1,
    "mu(1|23)^2" => 1,
    "mu(12|3)^2" => 1,
    "mu(13|2)^2" => 1,
    "mu(123)*mu(12|3)" => 1,
    "mu(123)*mu(13|2)" => 1,
    "mu(1|23)*mu(12|3)" => 1,
    "mu(1|23)*mu(13|2)" => 1,
    "mu(12|3)*mu(13|2)" => 2,
)

# Match each against enumerated rays
function find_ray(sig, rays)
    for (i, ray) in enumerate(rays)
        ray_sig = canonicalize_integer_ray(ray; normalize_sign=false)
        if ray_sig == sig
            return i
        end
    end
    return nothing
end

# Internal -> ECP label translation for display
const TO_ECP = Dict(
    "123"=>"abc", "1|2|3"=>"a|b|c", "1|23"=>"a|bc", "12|3"=>"ab|c", "13|2"=>"ac|b"
)

function to_ecp(s::AbstractString)
    result = s
    # Replace longer patterns first to avoid partial matches
    for (from, to) in sort(collect(TO_ECP), by=x->length(x[1]), rev=true)
        result = replace(result, from => to)
    end
    return result
end

named = [
    ("(5.4)",       sig_54,  "computer-assisted inequality"),
    ("(5.5)",       sig_55,  "Aas inequality"),
    ("Remark 5.5",  sig_r55, "third extremal ray"),
    ("(4.3)",       sig_43,  "derived, not extremal"),
]

println("Matching paper inequalities to extreme rays:\n")
println(rpad("Paper label", 14), rpad("Found?", 10), rpad("Ray #", 8), "Notes")
println("-"^60)

for (label, sig, note) in named
    idx = find_ray(sig, result_m4.rays)
    if idx !== nothing
        println(rpad(label, 14), rpad("yes", 10), rpad(string(idx), 8), note)
    else
        println(rpad(label, 14), rpad("no", 10), rpad("\u2014", 8), note)
    end
end

Matching paper inequalities to extreme rays:



Paper label   Found?    Ray #   Notes
------------------------------------------------------------
(5.4)         yes       16      computer-assisted inequality
(5.5)         yes       1       Aas inequality
Remark 5.5    yes       15      third extremal ray
(4.3)         no        —       derived, not extremal


In [8]:
println("\nMatched inequalities in ECP notation:\n")
for (label, sig, note) in named
    idx = find_ray(sig, result_m4.rays)
    idx === nothing && continue
    ecp_str = to_ecp(result_m4.formatted_inequalities[idx])
    println("  $label: $ecp_str")
end


Matched inequalities in ECP notation:

  (5.4): mu(a|bc)^2 - mu(abc)*mu(a|b|c) + mu(abc)*mu(ab|c) + mu(abc)*mu(ac|b) + mu(a|b|c)*mu(a|bc) + mu(a|bc)*mu(ab|c) + mu(a|bc)*mu(ac|b) + 2*mu(ab|c)*mu(ac|b) >= 0


  (5.5): mu(abc)*mu(a|b|c) - mu(a|bc)*mu(ab|c) - mu(a|bc)*mu(ac|b) - mu(ab|c)*mu(ac|b) >= 0
  Remark 5.5: -mu(abc)*mu(a|b|c) + mu(abc)*mu(a|bc) + mu(abc)*mu(ab|c) + mu(abc)*mu(ac|b) + mu(a|b|c)*mu(a|bc) + mu(a|bc)*mu(ab|c) + mu(a|bc)*mu(ac|b) + 2*mu(ab|c)*mu(ac|b) >= 0


## 8. Why (4.3) Is Not an Extreme Ray

The homogenized form of (4.3) is:

$$-xy + yz + z^2 + xu + xv + zu + zv + 2uv + u^2 + v^2 \ge 0$$

This equals (5.4) + $\mu(ab|c)^2 + \mu(ac|b)^2$. Since $\mu(ab|c)^2 \ge 0$ and
$\mu(ac|b)^2 \ge 0$ are both trivial positivity rays (extreme rays of the cone), (4.3)
is a non-negative combination of three extreme rays and therefore cannot itself be extremal.

## 9. Rays vs. Facets

The projected cone oracle returns both **extreme rays** and **facet normals**:

- Extreme rays of $D$ = valid universal percolation inequalities
  ($\sum \Phi(p,q)\,\mu(p)\mu(q) \ge 0$)
- Facet normals of $D$ = meta-constraints on the coefficient space

These are distinct objects from standard polyhedral duality. Inequalities come from
`result.rays`, not `result.facet_normals`.

In [9]:
println("Extreme rays (= inequalities): $(length(result_m4.rays))")
println("Facet normals (= meta-constraints): $(length(result_m4.facet_normals))")

Extreme rays (= inequalities): 17
Facet normals (= meta-constraints): 19


## 10. Computational Summary

In [10]:
println("Performance (HiGHS solver):\n")
println("  m  |   |F|  | rays | facets |  time (s) | iters | LP calls")
println("  ---|--------|------|--------|-----------|-------|--------")
for m in 1:4
    r = enumerate_quiet(3, m)
    pr = r.projection_result
    tuples_m = Set(Tuple(t[i] for i in 1:m) for t in tuples)
    @printf("  %d  | %5d  | %4d |   %4d | %9.2f | %5d | %6d\n",
        m, length(tuples_m), length(r.rays), length(r.facet_normals),
        r.timing, pr.iterations, pr.lp_calls)
end

Performance (HiGHS solver):

  m  |   |F|  | rays | facets |  time (s) | iters | LP calls
  ---|--------|------|--------|-----------|-------|--------


  1  |    25  |   15 |     15 |      0.06 |     1 |     15


  2  |   139  |   15 |     15 |      0.44 |     1 |     15


  3  |   570  |   16 |     16 |      4.76 |     1 |     16


  4  |  1265  |   17 |     19 |      3.77 |     2 |     21
